# 10a -- MTMC Stages 0-2: Frame Extraction + Detection + ReID Features

**Run once (or when tracking/ReID config changes). ~90 min on P100.**

| Stage | What | Time |
|---|---|---|
| 0 | Frame extraction (10 fps, 6 cameras) | ~20 min |
| 1 | YOLO detection + BotSort tracking | ~45 min |
| 2 | TransReID 768D + OSNet 512D -> PCA 256D features | ~20 min |

After this runs, its output (`checkpoint.tar.gz`) is used by **10b** -> **10c**.
Attach `mtmc-weights` via **Add Data -> Your Datasets -> mtmc-weights**.

In [ ]:
import os, sys, subprocess, shutil, json, time, tarfile
from pathlib import Path
import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({p.total_memory/1024**3:.1f} GB)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {DEVICE}")

## 1. Clone Repo & Install Dependencies

In [ ]:
REPO_URL = "https://github.com/MRKDaGods/gp.git"
WORK_DIR = Path("/kaggle/working")
PROJECT  = WORK_DIR / "gp"

if not PROJECT.exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)])
else:
    print("Repo already present, pulling latest ...")
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))
print(f"\n\u2713 Repo ready at {PROJECT}")

In [ ]:
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("ultralytics")
pip("boxmot")

try:
    import torchreid; print("torchreid ok")
except ImportError:
    pip("git+https://github.com/KaiyangZhou/deep-person-reid.git")

try:
    import faiss; print(f"faiss ok ({faiss.__version__})")
except ImportError:
    try: pip("faiss-gpu")
    except Exception: pip("faiss-cpu")

pip("timm", "motmetrics")
pip("gdown", "loguru", "omegaconf", "rich", "networkx>=3.1", "click")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"],
                      cwd=str(PROJECT))
print("\n\u2713 All dependencies installed")

In [ ]:
FAILED = []
_checks = [
    ("ultralytics", "ultralytics"),
    ("boxmot", "boxmot"),
    ("torch", "torch"),
    ("torchreid", "torchreid"),
    ("timm", "timm"),
    ("faiss", "faiss"),
    ("motmetrics", "motmetrics"),
    ("cv2", "cv2"),
    ("loguru", "loguru"),
    ("omegaconf", "omegaconf"),
    ("networkx", "networkx"),
    ("sklearn", "sklearn"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
]
for label, mod in _checks:
    try:
        __import__(mod)
        print(f"  \u2713 {label}")
    except ImportError as e:
        print(f"  \u2717 {label}: {e}")
        FAILED.append(label)
if FAILED:
    raise RuntimeError(f"Missing modules: {FAILED} -- fix pip installs above")
print("\n\u2713 All required modules importable")

## 2. Mount Model Weights
Model weights dataset (`mtmc-weights`) must be attached.

In [ ]:
WEIGHTS_INPUT = Path("/kaggle/input/datasets/mrkdagods/mtmc-weights/models")
assert WEIGHTS_INPUT.exists(), (
    "Dataset 'mtmc-weights' not found.\n"
    f"  Expected: {WEIGHTS_INPUT}\n"
    "  Attach via: Add Data -> Your Datasets -> mtmc-weights"
)

MODELS_DST = PROJECT / "models"
if MODELS_DST.is_symlink(): MODELS_DST.unlink()
if MODELS_DST.exists(): shutil.rmtree(MODELS_DST)
print(f"Copying models/ from {WEIGHTS_INPUT} (~750 MB) ...")
shutil.copytree(str(WEIGHTS_INPUT), str(MODELS_DST))

ESSENTIAL = [
    "models/detection/yolo26m.pt",
    "models/reid/transreid_cityflowv2_best.pth",
    "models/reid/vehicle_osnet_veri776.pth",
    "models/tracker/osnet_x0_25_msmt17.pt",
]
missing = [p for p in ESSENTIAL if not (PROJECT / p).exists()]
if missing:
    for m in missing: print(f"  MISSING: {m}")
    raise FileNotFoundError("Fix missing weights before continuing.")
print("\u2713 All essential v4 weights present")

## 3. Download CityFlowV2 (~17 GB)

Downloaded from Google Drive to `/tmp` (not `/kaggle/working` -- only 20 GB).
Peak disk in `/tmp`: ~44 GB (download + extraction + stage0 frames).

In [ ]:
import re as _re, shutil as _shutil

for mount in ["/tmp", "/kaggle/working"]:
    total, used, free = _shutil.disk_usage(mount)
    print(f"{mount:20s}  {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total")

CAM_RE = _re.compile(r"^S\d{2}_c\d{3}$")
TMP_DATA = Path("/tmp/datasets")
TMP_DATA.mkdir(parents=True, exist_ok=True)

DATA_RAW_PARENT = PROJECT / "data" / "raw"
if not DATA_RAW_PARENT.is_symlink():
    if DATA_RAW_PARENT.exists(): shutil.rmtree(DATA_RAW_PARENT)
    DATA_RAW_PARENT.parent.mkdir(parents=True, exist_ok=True)
    DATA_RAW_PARENT.symlink_to(TMP_DATA)
    print(f"\u2713 data/raw \u2192 {TMP_DATA}")
else:
    print(f"data/raw already symlinked -> {DATA_RAW_PARENT.resolve()}")

DATA_OUT = Path("/tmp/pipeline_outputs")
DATA_OUT.mkdir(parents=True, exist_ok=True)
DATA_RAW = TMP_DATA / "cityflowv2"

if DATA_RAW.exists() and any(CAM_RE.match(d.name) for d in DATA_RAW.iterdir() if d.is_dir()):
    cams = [d.name for d in DATA_RAW.iterdir() if d.is_dir() and CAM_RE.match(d.name)]
    print(f"\u2713 CityFlowV2 already present: {len(cams)} cameras")
else:
    print("Downloading CityFlowV2 from Google Drive (~17 GB) ...")
    subprocess.check_call(
        [sys.executable, "scripts/download_datasets.py", "--dataset", "cityflowv2"],
        cwd=str(PROJECT))
    cams = [d.name for d in DATA_RAW.iterdir() if d.is_dir() and CAM_RE.match(d.name)]
    print(f"\u2713 CityFlowV2 ready: {sorted(cams)}")
print(f"\nDataset path: {DATA_RAW}")

## 4. Run Stages 0-2

In [ ]:
from datetime import datetime
RUN_NAME = f"run_kaggle_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR  = DATA_OUT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
BENCHMARK_CAMERAS = ['S01_c001', 'S01_c002', 'S01_c003', 'S02_c006', 'S02_c007', 'S02_c008']
print(f"Run  : {RUN_NAME}")
print(f"Cams : {BENCHMARK_CAMERAS}")

In [ ]:
os.chdir(str(PROJECT))
cmd = [
    sys.executable, "scripts/run_pipeline.py",
    "--config", "configs/default.yaml",
    "--dataset-config", "configs/datasets/cityflowv2.yaml",
    "--stages", "0,1,2",
    "--override", f"project.run_name={RUN_NAME}",
    "--override", f"project.output_dir={DATA_OUT}",
    "--override", "stage0.cameras=[S01_c001,S01_c002,S01_c003,S02_c006,S02_c007,S02_c008]",
]
print("CMD:", " ".join(str(c) for c in cmd))
print("=" * 70)
t0 = time.time()
r = subprocess.run(cmd, cwd=str(PROJECT))
print("=" * 70)
elapsed = time.time() - t0
if r.returncode != 0:
    print(f"\u2717 FAILED after {elapsed/60:.1f} min"); sys.exit(r.returncode)
print(f"\u2713 Stages 0-2 done in {elapsed/60:.1f} min")

## 5. Save Checkpoint

Saves stage1 + stage2 outputs + GT annotations to `/kaggle/working/checkpoint.tar.gz`.
This file becomes the input for **10b**.
Stage0 frame images (~9.6 GB) are **not** included -- downstream stages do not need them.

In [ ]:
import re as _re2

CAM_RE2 = _re2.compile(r"^S\d{2}_c\d{3}$")
checkpoint_path = Path("/kaggle/working/checkpoint.tar.gz")
metadata_path   = Path("/kaggle/working/run_metadata.json")

with open(metadata_path, "w") as f:
    json.dump({"run_name": RUN_NAME}, f)

print(f"Building checkpoint for run: {RUN_NAME}")
with tarfile.open(str(checkpoint_path), "w:gz") as tar:
    tar.add(str(metadata_path), arcname="run_metadata.json")

    manifest = RUN_DIR / "stage0" / "frames_manifest.json"
    if manifest.exists():
        tar.add(str(manifest), arcname=f"{RUN_NAME}/stage0/frames_manifest.json")
        print("  + stage0/frames_manifest.json")

    for stage in ["stage1", "stage2"]:
        stage_dir = RUN_DIR / stage
        if stage_dir.exists():
            n = 0
            for fpath in stage_dir.rglob("*"):
                if fpath.is_file():
                    tar.add(str(fpath), arcname=f"{RUN_NAME}/{stage}/{fpath.relative_to(stage_dir)}")
                    n += 1
            print(f"  + {stage}/ ({n} files)")

    # GT annotation txt files needed by stage5 eval (small text files, not videos)
    gt_count = 0
    for cam_dir in DATA_RAW.iterdir():
        if cam_dir.is_dir() and CAM_RE2.match(cam_dir.name):
            gt_file = cam_dir / "gt" / "gt.txt"
            if gt_file.exists():
                tar.add(str(gt_file), arcname=f"gt_annotations/{cam_dir.name}/gt/gt.txt")
                gt_count += 1
    print(f"  + gt_annotations/ ({gt_count} GT files)")

size_mb = checkpoint_path.stat().st_size / 1024**2
print(f"\n\u2713 Checkpoint: {checkpoint_path}  ({size_mb:.1f} MB)")
print("  Next: attach this notebook's output to 10b, then push 10b.")